# 05 — Spillover Regression

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1******6

**Purpose:** Estimate whether US implied volatility smile parameters on day $t$ have predictive power for EU smile parameters on day $t+1$, using OLS with Newey-West heteroskedasticity- and autocorrelation-consistent (HAC) standard errors.

**Input:** `data/analysis_ready/panel_merged__<timestamp>.csv` (output of notebook 04)

**Outputs:**
- `data/outputs/regression_results__<timestamp>.csv` — coefficient table, all models
- `data/outputs/figures/` — smile parameter time series, residual diagnostics
- `logs/05_regression_log__<timestamp>.json`

---

## Regression specification

**Primary maturity node:** 30-day.  
Justification: highest liquidity in both equity option markets, primary node in Chen et al. (2022), and the maturity at which VIX is constructed. All other nodes (60, 91, 182, 365-day) are estimated as robustness checks.

**Three models — one per smile parameter:**

$$\text{ATM}^{EU}_{t+1} = \alpha + \beta_1 \text{ATM}^{US}_{t} + \beta_2 \Delta\text{VIX}_t + \beta_3 \text{ATM}^{EU}_{t} + \varepsilon_{t+1}$$

$$\text{Skew}^{EU}_{t+1} = \alpha + \beta_1 \text{Skew}^{US}_{t} + \beta_2 \Delta\text{VIX}_t + \beta_3 \text{Skew}^{EU}_{t} + \varepsilon_{t+1}$$

$$\text{Curvature}^{EU}_{t+1} = \alpha + \beta_1 \text{Curvature}^{US}_{t} + \beta_2 \Delta\text{VIX}_t + \beta_3 \text{Curvature}^{EU}_{t} + \varepsilon_{t+1}$$

**Regressors in each model:**
- $\text{Parameter}^{US}_t$ — same smile parameter from the US market on day $t$ (main variable of interest)
- $\Delta\text{VIX}_t = \text{VIX}_t - \text{VIX}_{t-1}$ — change in VIX (controls for global volatility shocks)
- $\text{Parameter}^{EU}_t$ — lagged EU parameter (controls for own-market persistence / autocorrelation)

**Standard errors:** Newey-West HAC with automatic lag selection (Newey and West, 1994).  
Justification: smile parameters are persistent time series; OLS standard errors will be biased downward. HAC correction is standard in this literature (Tompkins, 2001; Chen et al., 2022).

**Hypothesis of interest:** $\beta_1 > 0$ and statistically significant — US smile parameter predicts EU smile parameter after controlling for own-market persistence and global volatility.

---

## References
- Chen, J., Han, Q., Ryu, D. and Tang, J. (2022). Does the world smile together? *Journal of International Financial Markets, Institutions and Money*, 77, 101497.
- Malz, A.M. (1997). Estimating the probability distribution of the future exchange rate from option prices. *Journal of Derivatives*, 5(2), 18–36.
- Newey, W.K. and West, K.D. (1994). Automatic lag selection in covariance matrix estimation. *Review of Economic Studies*, 61(4), 631–653.
- Tompkins, R.G. (2001). Implied volatility surfaces: Uncovering regularities for options on financial futures. *European Journal of Finance*, 7(3), 198–230.

---
## Step 0 — Imports and configuration

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import json
import warnings
from datetime import datetime

import subprocess
subprocess.run(["pip", "install", "statsmodels"])
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller

import matplotlib
matplotlib.use('Agg')  # non-interactive backend — safe for Jupyter
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore', category=FutureWarning)

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

ANALYSIS_READY_DIR = '../data/analysis_ready/'
OUTPUT_DIR         = '../data/outputs/'
FIGURES_DIR        = '../data/outputs/figures/'
LOG_DIR            = '../logs/'

for d in [OUTPUT_DIR, FIGURES_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# Primary maturity node
PRIMARY_NODE = 30

# Robustness nodes (estimated after primary)
ROBUSTNESS_NODES = [60, 91, 182, 365]

# Smile parameters to model
PARAMETERS = ['atm_iv', 'skew', 'curvature']

# Newey-West lag selection: use int(4 * (T/100)^(2/9)) — standard rule of thumb
# Will be computed per regression once T is known

log = {
    'notebook': '05_regression',
    'run_timestamp': RUN_TIMESTAMP,
    'primary_node': PRIMARY_NODE,
    'robustness_nodes': ROBUSTNESS_NODES,
    'parameters': PARAMETERS,
    'steps': {}
}

print(f'Run timestamp: {RUN_TIMESTAMP}')
print(f'Primary maturity node: {PRIMARY_NODE}-day')
print(f'Robustness nodes: {ROBUSTNESS_NODES}-day')

Matplotlib is building the font cache; this may take a moment.


Run timestamp: 20260314_160035
Primary maturity node: 30-day
Robustness nodes: [60, 91, 182, 365]-day


---
## Step 1 — Load merged panel

In [2]:
pattern = os.path.join(ANALYSIS_READY_DIR, 'panel_merged__*.csv')
matches = glob.glob(pattern)

if len(matches) == 0:
    raise FileNotFoundError(
        f'No merged panel found at {pattern}. '
        'Run notebook 04 first.'
    )

latest = max(matches, key=os.path.getmtime)
panel = pd.read_csv(latest, parse_dates=['date_eu'])

print(f'Loaded: {latest}')
print(f'Shape: {panel.shape}')
print(f'Columns: {panel.columns.tolist()}')
print()

if len(panel) == 0:
    raise ValueError(
        'Merged panel has zero rows. '
        'EU and US sample windows do not overlap. '
        'Full OptionMetrics access is required before regression can run. '
        'See notebook 04 Step 3 and HANDOFF.md for data access options.'
    )

print(f'Date range: {panel["date_eu"].min().date()} to {panel["date_eu"].max().date()}')
print(f'Unique EU dates: {panel["date_eu"].nunique()}')
print(f'Maturity nodes present: {sorted(panel["days"].unique().tolist())}')

log['steps']['step1_load'] = {
    'file': latest,
    'shape': list(panel.shape),
    'date_min': str(panel['date_eu'].min().date()),
    'date_max': str(panel['date_eu'].max().date()),
    'n_dates': int(panel['date_eu'].nunique()),
    'nodes': sorted(panel['days'].unique().tolist()),
}

Loaded: ../data/analysis_ready/panel_merged__20260314_155525.csv
Shape: (0, 40)
Columns: ['date_eu', 'days', 'eu_atm_iv', 'eu_skew', 'eu_curvature', 'eu_put25_iv', 'eu_call75_iv', 'eu_us_rf_rate', 'eu_us_yield_1y', 'eu_us_yield_10y', 'eu_us_term_spread', 'eu_us_hy_spread', 'eu_sp500_ret', 'eu_sp500_idx', 'eu_ff_mktrf', 'eu_ff_smb', 'eu_ff_hml', 'eu_ff_rf', 'eu_vix', 'eu_eurusd', 'eu_hist_vol_30d', 'date', 'us_atm_iv', 'us_skew', 'us_curvature', 'us_put25_iv', 'us_call75_iv', 'us_us_rf_rate', 'us_us_yield_1y', 'us_us_yield_10y', 'us_us_term_spread', 'us_us_hy_spread', 'us_sp500_ret', 'us_sp500_idx', 'us_ff_mktrf', 'us_ff_smb', 'us_ff_hml', 'us_ff_rf', 'us_vix', 'us_eurusd']



ValueError: Merged panel has zero rows. EU and US sample windows do not overlap. Full OptionMetrics access is required before regression can run. See notebook 04 Step 3 and HANDOFF.md for data access options.

---
## Step 2 — Construct regression variables

Build the full variable set needed for all three models:
- ΔVIX: first difference of VIX (controls for global volatility shocks)
- Lagged EU parameters: own-market persistence control
- All constructed per maturity node

In [3]:
def build_regression_df(panel, node):
    """
    Given the full merged panel and a maturity node (days),
    returns a clean regression DataFrame for that node.

    Columns returned:
      date_eu           — EU observation date (dependent variable date = t+1)
      eu_atm_iv         — EU ATM IV at t+1  (dependent variable)
      eu_skew           — EU Skew at t+1    (dependent variable)
      eu_curvature      — EU Curvature at+1 (dependent variable)
      us_atm_iv         — US ATM IV at t     (main regressor)
      us_skew           — US Skew at t       (main regressor)
      us_curvature      — US Curvature at t  (main regressor)
      delta_vix         — ΔVIX at t          (control)
      eu_atm_iv_lag1    — EU ATM IV at t     (own-market persistence control)
      eu_skew_lag1      — EU Skew at t       (own-market persistence control)
      eu_curvature_lag1 — EU Curvature at t  (own-market persistence control)

    The lag-1 EU variables are constructed by shifting eu_ columns by 1
    within the node-specific slice, sorted by date.
    """
    df = panel[panel['days'] == node].copy()
    df = df.sort_values('date_eu').reset_index(drop=True)

    # ΔVIX: use eu_vix which is the EU-side VIX (same global series, just the
    # value on date t+1). We need VIX at t, which is us_vix in the merged panel.
    # ΔVIX_t = us_vix_t - us_vix_{t-1}
    df['delta_vix'] = df['us_vix'].diff()

    # Own-market persistence: EU parameter at t = lag-1 of EU parameter at t+1
    # Since panel is sorted by date_eu (= t+1), shift(1) gives t
    for param in PARAMETERS:
        df[f'eu_{param}_lag1'] = df[f'eu_{param}'].shift(1)

    # Drop first row (NaN from diff and shift)
    df = df.dropna(subset=['delta_vix'] + [f'eu_{p}_lag1' for p in PARAMETERS])
    df = df.reset_index(drop=True)

    print(f'Node {node}-day: {len(df)} observations after lag construction')
    return df


# Build for primary node
reg_primary = build_regression_df(panel, PRIMARY_NODE)

# Build for robustness nodes
reg_robustness = {}
for node in ROBUSTNESS_NODES:
    if node in panel['days'].unique():
        reg_robustness[node] = build_regression_df(panel, node)
    else:
        print(f'Node {node}-day not present in merged panel — skipping.')

print()
print(f'Primary regression dataset shape: {reg_primary.shape}')
print()
print('Sample (first 3 rows):')
display_cols = ['date_eu', 'eu_atm_iv', 'eu_skew', 'eu_curvature',
                'us_atm_iv', 'us_skew', 'us_curvature', 'delta_vix']
available = [c for c in display_cols if c in reg_primary.columns]
print(reg_primary[available].head(3).to_string())

log['steps']['step2_variables'] = {
    'primary_node_obs': len(reg_primary),
    'robustness_nodes_obs': {str(k): len(v) for k, v in reg_robustness.items()},
}

Node 30-day: 0 observations after lag construction
Node 60-day not present in merged panel — skipping.
Node 91-day not present in merged panel — skipping.
Node 182-day not present in merged panel — skipping.
Node 365-day not present in merged panel — skipping.

Primary regression dataset shape: (0, 44)

Sample (first 3 rows):
Empty DataFrame
Columns: [date_eu, eu_atm_iv, eu_skew, eu_curvature, us_atm_iv, us_skew, us_curvature, delta_vix]
Index: []


---
## Step 3 — Pre-regression diagnostics

Before estimating, run two standard checks:

1. **ADF unit root test** on each dependent variable — if a series has a unit root, OLS on levels is spurious. We test and report; if unit root is found, we switch to first differences for that variable and flag it.
2. **Summary statistics** for all regression variables.

In [4]:
print('=== PRE-REGRESSION DIAGNOSTICS (PRIMARY NODE: 30-day) ===')
print()

# Summary statistics
reg_cols = (
    [f'eu_{p}' for p in PARAMETERS] +
    [f'us_{p}' for p in PARAMETERS] +
    ['delta_vix'] +
    [f'eu_{p}_lag1' for p in PARAMETERS]
)
available_reg_cols = [c for c in reg_cols if c in reg_primary.columns]

print('Summary statistics:')
print(reg_primary[available_reg_cols].describe().round(6).to_string())
print()

# ADF unit root tests on dependent variables
# H0: series has a unit root (non-stationary)
# If p-value > 0.05: cannot reject unit root → flag for differencing
print('ADF unit root tests (H0: unit root present):')
print(f'{"Variable":<25} {"ADF stat":>10} {"p-value":>10} {"Decision":>20}')
print('-' * 70)

adf_results = {}
unit_root_flags = []

for param in PARAMETERS:
    col = f'eu_{param}'
    if col not in reg_primary.columns:
        continue
    series = reg_primary[col].dropna()
    if len(series) < 10:
        print(f'{col:<25} {"N/A":>10} {"N/A":>10} {"Too few obs":>20}')
        continue
    result = adfuller(series, autolag='AIC')
    adf_stat = result[0]
    p_val    = result[1]
    decision = 'Stationary ✓' if p_val <= 0.05 else 'Unit root — FLAG'
    if p_val > 0.05:
        unit_root_flags.append(col)
    adf_results[col] = {'adf_stat': round(adf_stat, 4), 'p_value': round(p_val, 4)}
    print(f'{col:<25} {adf_stat:>10.4f} {p_val:>10.4f} {decision:>20}')

print()
if unit_root_flags:
    print(f'Unit root flagged in: {unit_root_flags}')
    print('These variables will be first-differenced in their respective regressions.')
    print('Results for both levels and differences will be reported for transparency.')
else:
    print('No unit roots detected. Proceeding with levels.')

log['steps']['step3_diagnostics'] = {
    'adf_results': adf_results,
    'unit_root_flags': unit_root_flags,
}

=== PRE-REGRESSION DIAGNOSTICS (PRIMARY NODE: 30-day) ===

Summary statistics:
       eu_atm_iv eu_skew eu_curvature us_atm_iv us_skew us_curvature delta_vix eu_atm_iv_lag1 eu_skew_lag1 eu_curvature_lag1
count          0       0            0         0       0            0         0              0            0                 0
unique         0       0            0         0       0            0         0              0            0                 0
top          NaN     NaN          NaN       NaN     NaN          NaN       NaN            NaN          NaN               NaN
freq         NaN     NaN          NaN       NaN     NaN          NaN       NaN            NaN          NaN               NaN

ADF unit root tests (H0: unit root present):
Variable                    ADF stat    p-value             Decision
----------------------------------------------------------------------
eu_atm_iv                        N/A        N/A          Too few obs
eu_skew                          N/A     

---
## Step 4 — Estimate primary regressions (30-day node)

Three separate OLS regressions, one per smile parameter.  
Standard errors: Newey-West HAC, lag = $\lfloor 4(T/100)^{2/9} \rfloor$ (Newey and West, 1994 automatic rule).

Each model is:
$$Y^{EU}_{t+1} = \alpha + \beta_1 Y^{US}_t + \beta_2 \Delta\text{VIX}_t + \beta_3 Y^{EU}_t + \varepsilon_{t+1}$$

where $Y$ is ATM IV, Skew, or Curvature respectively.

In [5]:
def newey_west_lags(T):
    """
    Newey-West (1994) automatic lag selection rule.
    lag = floor(4 * (T/100)^(2/9))
    """
    return int(np.floor(4 * (T / 100) ** (2 / 9)))


def run_ols_nw(df, dep_var, indep_vars, label):
    """
    Run OLS with Newey-West HAC standard errors.

    Parameters
    ----------
    df          : DataFrame containing all variables
    dep_var     : str, name of dependent variable column
    indep_vars  : list of str, names of independent variable columns
    label       : str, label for printing

    Returns
    -------
    results     : statsmodels RegressionResultsWrapper
    summary_dict: dict of key results for logging
    """
    # Drop any remaining NaNs in the variables used
    cols_needed = [dep_var] + indep_vars
    clean = df[cols_needed].dropna()
    T = len(clean)

    if T < 10:
        print(f'{label}: insufficient observations (T={T}). Skipping.')
        return None, None

    y = clean[dep_var]
    X = sm.add_constant(clean[indep_vars])

    # OLS fit
    model  = sm.OLS(y, X)
    # Newey-West HAC covariance
    nw_lags = newey_west_lags(T)
    results = model.fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})

    # Durbin-Watson for residual autocorrelation diagnostics
    dw_stat = durbin_watson(results.resid)

    print(f'--- {label} (T={T}, NW lags={nw_lags}) ---')
    print(f'R-squared: {results.rsquared:.4f}   Adj R-squared: {results.rsquared_adj:.4f}')
    print(f'Durbin-Watson: {dw_stat:.4f} (2.0 = no autocorrelation)')
    print()
    print(f'{"Variable":<25} {"Coef":>10} {"NW SE":>10} {"t-stat":>10} {"p-value":>10}')
    print('-' * 70)
    for var in results.params.index:
        coef   = results.params[var]
        se     = results.bse[var]
        tstat  = results.tvalues[var]
        pval   = results.pvalues[var]
        stars  = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f'{var:<25} {coef:>10.6f} {se:>10.6f} {tstat:>10.4f} {pval:>10.4f} {stars}')
    print()

    summary_dict = {
        'T': T,
        'nw_lags': nw_lags,
        'r_squared': round(results.rsquared, 6),
        'adj_r_squared': round(results.rsquared_adj, 6),
        'durbin_watson': round(dw_stat, 4),
        'coefficients': {
            v: {
                'coef':   round(results.params[v], 8),
                'se_nw':  round(results.bse[v], 8),
                'tstat':  round(results.tvalues[v], 4),
                'pvalue': round(results.pvalues[v], 6),
            } for v in results.params.index
        }
    }

    return results, summary_dict


# Run three primary regressions
print('====== PRIMARY REGRESSIONS — 30-day node ======')
print()

primary_results = {}
primary_summaries = {}

for param in PARAMETERS:
    dep   = f'eu_{param}'
    indep = [f'us_{param}', 'delta_vix', f'eu_{param}_lag1']
    indep_avail = [c for c in indep if c in reg_primary.columns]

    label = f'Model: EU {param} (t+1) ~ US {param} (t) + controls'
    res, summ = run_ols_nw(reg_primary, dep, indep_avail, label)
    primary_results[param]   = res
    primary_summaries[param] = summ

log['steps']['step4_primary_regressions'] = {
    'node': PRIMARY_NODE,
    'results': primary_summaries
}

====== PRIMARY REGRESSIONS — 30-day node ======

Model: EU atm_iv (t+1) ~ US atm_iv (t) + controls: insufficient observations (T=0). Skipping.
Model: EU skew (t+1) ~ US skew (t) + controls: insufficient observations (T=0). Skipping.
Model: EU curvature (t+1) ~ US curvature (t) + controls: insufficient observations (T=0). Skipping.


---
## Step 5 — Robustness checks: other maturity nodes

Repeat the same three models for each robustness node (60, 91, 182, 365-day). The primary conclusion should be robust to maturity choice if the spillover is a genuine cross-market phenomenon and not specific to the 30-day contract.

In [6]:
robustness_log = {}

for node, df_node in reg_robustness.items():
    print(f'====== ROBUSTNESS — {node}-day node ======')
    print()
    node_summaries = {}
    for param in PARAMETERS:
        dep   = f'eu_{param}'
        indep = [f'us_{param}', 'delta_vix', f'eu_{param}_lag1']
        indep_avail = [c for c in indep if c in df_node.columns]
        label = f'{node}-day | EU {param} ~ US {param} + controls'
        _, summ = run_ols_nw(df_node, dep, indep_avail, label)
        node_summaries[param] = summ
    robustness_log[str(node)] = node_summaries

log['steps']['step5_robustness'] = robustness_log

---
## Step 6 — Residual diagnostics (primary regressions only)

For each primary regression, plot:
1. Residuals over time — should show no pattern
2. Residual histogram — should be approximately normal

These are standard diagnostic checks. Non-normality is not a fatal flaw for OLS (CLT applies for large T) but severe autocorrelation would indicate the Newey-West lag selection may be insufficient.

In [7]:
for param in PARAMETERS:
    res = primary_results.get(param)
    if res is None:
        print(f'No result for {param} — skipping diagnostic plot.')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(
        f'Residual diagnostics — {param.upper()} model (30-day)',
        fontsize=11, fontweight='bold'
    )

    # Get the dates aligned with residuals
    dep = f'eu_{param}'
    indep = [f'us_{param}', 'delta_vix', f'eu_{param}_lag1']
    indep_avail = [c for c in indep if c in reg_primary.columns]
    clean = reg_primary[[dep] + indep_avail].dropna()

    # Plot 1: Residuals over time
    axes[0].plot(clean.index, res.resid, color='steelblue', linewidth=0.8)
    axes[0].axhline(0, color='black', linewidth=0.6, linestyle='--')
    axes[0].set_title('Residuals over time')
    axes[0].set_xlabel('Observation index')
    axes[0].set_ylabel('Residual')

    # Plot 2: Residual histogram
    axes[1].hist(res.resid, bins=20, color='steelblue',
                 edgecolor='white', alpha=0.8)
    axes[1].set_title('Residual distribution')
    axes[1].set_xlabel('Residual')
    axes[1].set_ylabel('Frequency')

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, f'diagnostics_{param}_30day__{RUN_TIMESTAMP}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
    print()

log['steps']['step6_diagnostics'] = {'figures_saved_to': FIGURES_DIR}

No result for atm_iv — skipping diagnostic plot.
No result for skew — skipping diagnostic plot.
No result for curvature — skipping diagnostic plot.


---
## Step 7 — Smile parameter time series plots

Plot EU and US smile parameters over time for the primary node.  
These plots go directly into the thesis descriptive statistics section.

In [8]:
param_labels = {
    'atm_iv':    'ATM Implied Volatility',
    'skew':      'Skew [put(Δ25) − call(Δ75)]',
    'curvature': 'Curvature [put(Δ25) + call(Δ75) − 2×ATM]'
}

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    f'EU vs US Smile Parameters — {PRIMARY_NODE}-day maturity',
    fontsize=12, fontweight='bold'
)

for i, param in enumerate(PARAMETERS):
    ax = axes[i]
    eu_col = f'eu_{param}'
    us_col = f'us_{param}'

    if eu_col in reg_primary.columns:
        ax.plot(
            reg_primary['date_eu'], reg_primary[eu_col],
            label='EU (t+1)', color='steelblue', linewidth=1.2
        )
    if us_col in reg_primary.columns:
        ax.plot(
            reg_primary['date_eu'], reg_primary[us_col],
            label='US (t)', color='darkorange', linewidth=1.2, linestyle='--'
        )

    ax.set_ylabel(param_labels.get(param, param), fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[-1].set_xlabel('Date', fontsize=9)
plt.xticks(rotation=30)
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, f'smile_params_timeseries__{RUN_TIMESTAMP}.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

Saved: ../data/outputs/figures/smile_params_timeseries__20260314_160035.png


/var/folders/yn/ckwt_v6d45v5049z1mf7t11h0000gn/T/ipykernel_13527/26268430.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Step 8 — Save results table and log

In [9]:
# Build a flat results DataFrame for easy export
rows = []

for param, summ in primary_summaries.items():
    if summ is None:
        continue
    for var, stats in summ['coefficients'].items():
        rows.append({
            'node':        PRIMARY_NODE,
            'model':       param,
            'variable':    var,
            'coef':        stats['coef'],
            'se_nw':       stats['se_nw'],
            'tstat':       stats['tstat'],
            'pvalue':      stats['pvalue'],
            'r_squared':   summ['r_squared'],
            'adj_r_squared': summ['adj_r_squared'],
            'T':           summ['T'],
            'nw_lags':     summ['nw_lags'],
            'durbin_watson': summ['durbin_watson'],
            'specification': 'primary'
        })

for node_str, node_summs in robustness_log.items():
    for param, summ in node_summs.items():
        if summ is None:
            continue
        for var, stats in summ['coefficients'].items():
            rows.append({
                'node':        int(node_str),
                'model':       param,
                'variable':    var,
                'coef':        stats['coef'],
                'se_nw':       stats['se_nw'],
                'tstat':       stats['tstat'],
                'pvalue':      stats['pvalue'],
                'r_squared':   summ['r_squared'],
                'adj_r_squared': summ['adj_r_squared'],
                'T':           summ['T'],
                'nw_lags':     summ['nw_lags'],
                'durbin_watson': summ['durbin_watson'],
                'specification': 'robustness'
            })

results_df = pd.DataFrame(rows)

# Save
results_path = os.path.join(OUTPUT_DIR, f'regression_results__{RUN_TIMESTAMP}.csv')
log_path     = os.path.join(LOG_DIR, f'05_regression_log__{RUN_TIMESTAMP}.json')

results_df.to_csv(results_path, index=False)
print(f'Results table saved: {results_path}')
print(f'Rows: {len(results_df)}')
print()
print(results_df.to_string())

log['output_file'] = results_path
log['status'] = 'complete'

with open(log_path, 'w') as f:
    json.dump(log, f, indent=2, default=str)
print()
print(f'Log saved: {log_path}')

Results table saved: ../data/outputs/regression_results__20260314_160035.csv
Rows: 0

Empty DataFrame
Columns: []
Index: []

Log saved: ../logs/05_regression_log__20260314_160035.json


---
## Step 9 — Interpretation guide

Use this cell as a reference when writing the results section of the thesis.

**What to report for each model:**
- Coefficient on `us_<param>` ($\hat{\beta}_1$), its Newey-West standard error, t-statistic, and p-value
- $R^2$ and adjusted $R^2$
- Sample size $T$ and number of Newey-West lags used
- Durbin-Watson statistic (near 2.0 = acceptable; far from 2.0 = residual autocorrelation remains)

**How to interpret $\hat{\beta}_1$:**
- $\hat{\beta}_1 > 0$, $p < 0.05$: US smile parameter on day $t$ predicts EU smile parameter on day $t+1$, consistent with the spillover hypothesis.
- $\hat{\beta}_1 \approx 0$ or $p > 0.10$: no statistically detectable spillover in this parameter/maturity combination.
- Never say "proves" or "confirms" — say "is consistent with" or "suggests evidence of".

**Limitations to state explicitly in the thesis:**
1. Results are based on sample data (Adidas EU, Apple US) — not SPX/STOXX 50 index options. Generalisability is limited.
2. Short sample window — coefficient estimates have high uncertainty. Results should be interpreted as indicative, not conclusive.
3. OLS assumes linearity. Non-linear spillovers (e.g., threshold effects during crises) are not captured.
4. No causal identification — the lag structure is motivated by time zones, but omitted variables (e.g., overnight news) may confound the estimate.